# r3con — options

Every setting, over the same five memos. The correct answer throughout is **Halloran
Services Ltd, 11**.


In [ ]:
from r3con import r3con

QUESTION = ("Which contractor was responsible for the most equipment incidents across our "
            "sites in Q3, and how many? Give the contractor's name, not its code.")
docs = r3con.read_documents("memos")

MODEL = "hosted_vllm/Qwen/Qwen3.5-35B-A3B"   # any litellm model string
BASE  = "http://localhost:8555/v1"           # only for a self-hosted endpoint

## What shapes the answer


In [ ]:
result = r3con.run(
    QUESTION, docs,
    model=MODEL, api_base=BASE,
    relevance_rounds=2,           # times each document is re-read. DEFAULT: 2.
                                  #   1 = each document read against the question alone
                                  #   2+ = re-read in light of the other documents' snippets
    seed=42,                      # default 42
    params={"temperature": 0.3},  # passed straight to litellm.completion: top_p,
)                                 #   max_tokens, extra_body, anything it accepts
print(result)

Halloran Services Ltd was responsible for the most equipment incidents in Q3, with a total of 11 incidents across the Northgate and Riverside sites.


## Run configs — the same settings, named and reusable


In [ ]:
from r3con import load_config

cfg = load_config("default")
print(cfg.label())        # the run identity: two runs are the same run iff these match

# Override fields, or add your own configs/<name>.yaml and select it by name.
print(load_config("default", model=MODEL, relevance_rounds=3).label())

# A RunConfig can be passed straight to run(config=...). Everything that shapes the
# output lives here; api_base / api_key / completion are transport and stay out of it.

default[model=gpt-5.6-luna,seed=42,rounds=2,prompts=(rel=v1,schema=v1,parse=v1,reason=v1)]
default[model=Qwen3.5-35B-A3B,seed=42,rounds=3,prompts=(rel=v1,schema=v1,parse=v1,reason=v1)]


## Your own connection — what `completion=` is for


In [ ]:
import litellm

# `completion=` takes any callable shaped like litellm.completion. A litellm Router's
# .completion is one: it load-balances and falls back across deployments. Use it when you
# already own the connection — a gateway, a cache, a wrapper that logs.
router = litellm.Router(model_list=[
    {"model_name": "qwen",
     "litellm_params": {"model": MODEL, "api_base": BASE}},
    {"model_name": "qwen",
     "litellm_params": {"model": MODEL, "api_base": BASE}},
])

# With a Router the model string is the router's model_name, not the litellm one.
routed = r3con.run(QUESTION, docs, model="qwen", completion=router.completion)
print(routed)

Halloran Services Ltd was responsible for the most equipment incidents across our sites in Q3, with a total of 11 incidents (5 at Northgate and 6 at Riverside).


## Where the artifacts go


In [ ]:
import os

print(os.path.relpath(result.run_dir))     # under ./logs by default; logs_dir=... moves it,
for p in sorted(result.run_dir.rglob("*.json")):   # or set R3CON_LOGS_DIR
    print("  ", p.relative_to(result.run_dir))

# r3con.run(..., save_artifacts=False) writes nothing at all.

logs/20260921T181837Z_80eee7aa
   manifest.json
   reasoning/calls.json
   reasoning/result.json
   relevance/calls.json
   relevance/result.json
   structuring/parsing/calls.json
   structuring/parsing/result.json
   structuring/schema/calls.json
   structuring/schema/result.json
